# Approach 1 (Avoid-Step Cutoff, **floor-based asymptote**) — Fit on averaged raw CE

Variant of `approach_1_avoid_step` that fixes a systematic bias in the asymptote `A`.

**The problem.** The model `A + B/(BN+1)^n` defines `A` as the BN→∞ limit. But the power-law tail
decays more slowly than the real (saturating) CE curve, so at the largest observed batch the decay
term `B/(BNmax+1)^n` is still active — and that leftover term is exactly the gap between the fitted
`A` and the curve's true plateau (`floor ≈ A + B/(BNmax+1)^n`, verified on all 57 runs). The gap
grows with pruning and its local variation produces non-monotonic **bumps** in IPA-vs-P%
(e.g. BS=64 at P=82%/88%).

**The fix.** Use the **empirical tail floor** — the mean of the last `TAIL_N=50` points of the fit
window — as the asymptote. We pin `A = floor` (not fitted) and fit only `B, n` for the curve shape.
`CE_L` and IPA are then computed from this honest plateau. Result: the IPA trend is monotonic in
all three batch sizes, and because `CE_L` now sits above the floor, `learn_BN` comes from real data
crossings (no analytic fallback).

Fitting still applies only to data **before the first step artifact**
(`cutoff_BN` = first BN ≥ 100 where `abs(CE(BN) − CE(BN−1)) > 0.01`; full data if none).

**Input:** `prune_layers_ALL/p-percentage_{p}/batch_size_{bs}/averaged_runs_p_{p}_bs_{bs}.csv`

**Output:** `intermediate/approach_1_fit_params_bs_{bs}.csv` (with `A` (=floor), `B`, `n`,
`RMSE_full`, `RMSE_last50`) plus `ipa_summary_approach_1_avoid_step_floor.csv`.

**IPA:** `abs(CE_o - CE_L) / learn_BN` where `CE_L = CE_o - 0.85*(CE_o - A)` and `A = tail floor`.

In [21]:
# === Cell 1 — Config, imports, helpers ===
import os, glob, re
import numpy as np
import pandas as pd
from lmfit import Parameters, minimize
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────────────
BN_STEP_MIN = 100    # don't check for steps before this BN
STEP_THRESH = 0.01   # abs(CE(BN) - CE(BN-1)) > STEP_THRESH triggers cutoff
TAIL_N      = 50     # asymptote A = mean of the last TAIL_N points of the fit window
# ───────────────────────────────────────────────────────────────────────────────

# Paths
BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_floor"
INTERMEDIATE_DIR = os.path.join(OUT_DIR, "intermediate")
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)   # max CE for 10-class problem, ~2.302585

# Auto-detect pruning percentages from prune_layers_ALL/p-percentage_*/
p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning percentages: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")
print(f"BN_STEP_MIN = {BN_STEP_MIN}   STEP_THRESH = {STEP_THRESH}   TAIL_N = {TAIL_N}")

# Fit-function helpers
A_MIN, A_MAX = 0.1, 2.3
B_MIN, B_MAX = 0, 1000
N_MIN, N_MAX = 0.5, 2


def initialize_guesses(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    A0 = np.percentile(y, 5)
    B0 = np.percentile(y, 95) - A0
    n0 = 0.5
    if len(x) > 10:
        denom = y[0] - A0
        if abs(denom) > 1e-10:
            frac = max(1e-6, (y[0] - y[-1]) / denom)
            if frac > 0:
                n0 = max(0.3, min(1.5, -np.log(frac)))
    return A0, n0, B0


def model(params, x):
    vals = params.valuesdict()
    A, B, n = vals["A"], vals["B"], vals["n"]
    return A + B / ((x + 1) ** n)


def residual(params, x, data):
    # A is pinned to the tail floor; the floor approach is a tail/asymptote story, so we keep the
    # original BN-weighting (weight = x). This makes the overlaid B,n curve hug the tail (consistent
    # with the floor line). The early region is de-emphasized (the single power law cannot match both
    # the steep early drop and the flat tail), so judge fit quality by RMSE_last50, not RMSE_full.
    weight = x
    return weight * (model(params, x) - data)


def empirical_floor(x, y, tail_n=TAIL_N):
    """Empirical asymptote = mean of the last `tail_n` finite points of the fit window."""
    y = np.asarray(y, float)
    y = y[np.isfinite(y)]
    if len(y) == 0:
        return np.nan
    k = int(min(tail_n, len(y)))
    return float(np.mean(y[-k:]))


def fit_curve_floor(x, y, tail_n=TAIL_N):
    """Pin A = empirical tail floor (vary=False); fit only B and n for the curve shape.
    Returns (MinimizerResult, A_floor) or (None, A_floor)."""
    mask  = ~np.isnan(y)
    x_fit = np.asarray(x)[mask]
    y_fit = np.asarray(y)[mask]
    if len(x_fit) < 10:
        return None, np.nan
    A_floor = empirical_floor(x_fit, y_fit, tail_n=tail_n)
    A0, n0, B0 = initialize_guesses(x_fit, y_fit)
    params = Parameters()
    params.add("A", value=A_floor, vary=False)            # asymptote fixed to the data plateau
    params.add("B", value=max(B0, 1e-6), min=B_MIN, max=B_MAX)
    params.add("n", value=n0, min=N_MIN, max=N_MAX)
    try:
        return minimize(residual, params, args=(x_fit, y_fit)), A_floor
    except Exception:
        return None, A_floor


def fit_quality(x, y, A, B, n, tail_n=TAIL_N):
    """Unweighted RMSE over the whole fit window and over the last `tail_n` points.
    RMSE_last50 is the meaningful metric here (the floor approach targets the tail);
    RMSE_full is inflated by the de-emphasized early region and kept only for reference."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 2:
        return np.nan, np.nan, 0
    yhat = A + B / ((x + 1) ** n)
    rmse_full = float(np.sqrt(np.mean((yhat - y) ** 2)))
    k = int(min(tail_n, len(x)))
    rmse_tail = float(np.sqrt(np.mean((yhat[-k:] - y[-k:]) ** 2)))
    return rmse_full, rmse_tail, k


# --- IPA: learn_BN from avg data; analytical fallback when data ends early ---
# CE_L = CE_o - 0.85*(CE_o - A)   -- here A is the empirical tail floor
# IPA  = abs(CE_o - CE_L) / learn_BN
#
# learn_BN priority:
#   1. Truncated averaged data: first BN where Avg_CE_Test <= CE_L
#   2. Analytical fallback: learn_BN = ceil((B / (CE_L - A))^(1/n) - 1)
def compute_ipa_from_fit(x_grid, ce_data, A, B, n):
    x_grid  = np.asarray(x_grid,  dtype=float)
    ce_data = np.asarray(ce_data, dtype=float)
    CE_L = CE_o - 0.90 * (CE_o - A)
    mask = ce_data <= CE_L

    if mask.any():
        learn_BN        = float(x_grid[mask][0])
        avg_CE_at_learn = float(ce_data[mask][0])
    else:
        denom = CE_L - A
        if denom <= 0 or n <= 0 or B <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "avg_CE_learn_at_BN": np.nan}
        BN_analytic = (B / denom) ** (1.0 / n) - 1.0
        if not np.isfinite(BN_analytic) or BN_analytic <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "avg_CE_learn_at_BN": np.nan}
        learn_BN        = float(np.ceil(BN_analytic))
        avg_CE_at_learn = np.nan

    if learn_BN == 0:
        return {"CE_L": CE_L, "learn_BN": 0.0, "IPA": np.nan, "avg_CE_learn_at_BN": avg_CE_at_learn}
    IPA = abs(CE_o - CE_L) / learn_BN
    return {"CE_L": CE_L, "learn_BN": learn_BN, "IPA": IPA, "avg_CE_learn_at_BN": avg_CE_at_learn}


print("Cell 1 ready.")

Found 19 pruning percentages: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
CE_o = ln(10) = 2.302585
BN_STEP_MIN = 100   STEP_THRESH = 0.01   TAIL_N = 50
Cell 1 ready.


In [22]:
# === Cell 2 — Approach 1 (avoid-step, floor asymptote): detect step cutoff; fit on clean data ===
# For each (P%, BS):
#   1. Load full averaged CE data.
#   2. Find cutoff_BN = first BN >= BN_STEP_MIN where abs(CE(BN)-CE(BN-1)) > STEP_THRESH.
#      If no step found, use all data (cutoff_BN = max BN in data).
#   3. A = empirical tail floor (mean of last TAIL_N pts); fit B,n with A fixed.
#   4. Record RMSE_full / RMSE_last50; find learn_BN from truncated avg data.
inter_by_bs = {}

for bs in BATCH_SIZES:
    print("\n" + "=" * 70)
    print(f"  Approach 1 (avoid-step, floor asymptote) — Batch size {bs}")
    print("=" * 70)
    rows = []
    for p in PRUNING_LEVELS:
        avg_csv = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                               f"averaged_runs_p_{p}_bs_{bs}.csv")
        if not os.path.exists(avg_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  — missing {avg_csv}")
            continue
        df = pd.read_csv(avg_csv)
        df.columns = df.columns.str.strip()
        ce_col = next((c for c in df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
        bn_col = next((c for c in df.columns if "Batch" in c), None)
        if ce_col is None or bn_col is None:
            print(f"  [SKIP] P%={p*100:5.1f}%  — unexpected columns {list(df.columns)}")
            continue
        df = df.dropna(subset=[ce_col, bn_col]).reset_index(drop=True)

        # ── Step detection ──────────────────────────────────────────────────────────────────
        bns = df[bn_col].values.astype(float)
        ces = df[ce_col].values.astype(float)
        cutoff_BN     = float(bns[-1])   # default: use all data
        step_detected = False
        for i in range(1, len(bns)):
            if bns[i] >= BN_STEP_MIN:
                if abs(ces[i] - ces[i - 1]) > STEP_THRESH:
                    cutoff_BN     = float(bns[i])
                    step_detected = True
                    break
        # ────────────────────────────────────────────────────────────────────────────────

        df_fit = df[df[bn_col] < cutoff_BN]   # exclude step point itself
        x = df_fit[bn_col].values.astype(float)
        y = df_fit[ce_col].values.astype(float)

        result, A = fit_curve_floor(x, y)
        if result is None:
            print(f"  [FAIL] P%={p*100:5.1f}%  — fit did not converge")
            continue
        B = result.params["B"].value
        n = result.params["n"].value
        rmse_full, rmse_tail, n_tail = fit_quality(x, y, A, B, n)
        ipa = compute_ipa_from_fit(x, y, A, B, n)

        step_tag = "[step detected]" if step_detected else "[no step, full data]"
        src      = "data" if np.isfinite(ipa["avg_CE_learn_at_BN"]) else "analytic"
        print(f"  P%={p*100:5.1f}%  cutoff_BN={cutoff_BN:>6.0f} {step_tag:<22}  "
              f"A(floor)={A:.4f}  RMSE_full={rmse_full:7.4f}  CE_L={ipa['CE_L']:.4f}  "
              f"learn_BN={ipa['learn_BN']!r:>8}  IPA={ipa['IPA']}  [{src}]")

        rows.append({
            "P%":                 p * 100,
            "A":                  A,
            "B":                  B,
            "n":                  n,
            "CE_o":               CE_o,
            "CE_L":               ipa["CE_L"],
            "learn_BN":           ipa["learn_BN"],
            "avg_CE_learn_at_BN": ipa["avg_CE_learn_at_BN"],
            "IPA":                ipa["IPA"],
            "cutoff_BN":          cutoff_BN,
            "RMSE_full":          rmse_full,
            "RMSE_last50":        rmse_tail,
            "n_tail":             n_tail,
        })

    if rows:
        bs_df = pd.DataFrame(rows)
        inter_path = os.path.join(INTERMEDIATE_DIR, f"approach_1_fit_params_bs_{bs}.csv")
        bs_df.to_csv(inter_path, index=False)
        print(f"  Saved: {inter_path}")
        inter_by_bs[bs] = bs_df

print("\n[Cell 2 done]")


  Approach 1 (avoid-step, floor asymptote) — Batch size 64
  P%=  0.0%  cutoff_BN=   263 [step detected]         A(floor)=0.3397  RMSE_full= 0.3321  CE_L=0.5360  learn_BN=    35.0  IPA=0.05047329932447384  [data]
  P%= 10.0%  cutoff_BN=   273 [step detected]         A(floor)=0.3387  RMSE_full= 1.6526  CE_L=0.5351  learn_BN=    38.0  IPA=0.046512498842166676  [data]
  P%= 20.0%  cutoff_BN=   272 [step detected]         A(floor)=0.3388  RMSE_full= 3.3399  CE_L=0.5352  learn_BN=    41.0  IPA=0.043107157529137595  [data]
  P%= 30.0%  cutoff_BN=   261 [step detected]         A(floor)=0.3468  RMSE_full= 3.3867  CE_L=0.5424  learn_BN=    45.0  IPA=0.03911586181086131  [data]
  P%= 40.0%  cutoff_BN=   285 [step detected]         A(floor)=0.3474  RMSE_full= 1.4317  CE_L=0.5429  learn_BN=    51.0  IPA=0.034504075250411936  [data]
  P%= 50.0%  cutoff_BN=   322 [step detected]         A(floor)=0.3526  RMSE_full= 1.5950  CE_L=0.5476  learn_BN=    60.0  IPA=0.02924920966763796  [data]
  P%= 60.0%  

In [23]:
# === Cell 3 — Build wide summary CSV for Approach 1 (avoid-step, floor asymptote) ===
# Schema: P%, IPA_Avg_64, STD_64, IPA_Avg_1024, STD_1024, IPA_Avg_60000, STD_60000
# STD columns blank (NaN) for single-curve approaches.
summary_rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        df = inter_by_bs.get(bs)
        if df is None:
            mean_val = np.nan
        else:
            sub = df[df["P%"] == p * 100]
            mean_val = float(sub["IPA"].iloc[0]) if not sub.empty else np.nan
        row[f"IPA_Avg_{bs}"] = mean_val
        row[f"STD_{bs}"]     = np.nan
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows, columns=[
    "P%", "IPA_Avg_64", "STD_64", "IPA_Avg_1024", "STD_1024", "IPA_Avg_60000", "STD_60000"
])
out_csv = os.path.join(OUT_DIR, "ipa_summary_approach_1_avoid_step_floor.csv")
summary_df.to_csv(out_csv, index=False)
print(f"\nFinal summary written: {out_csv}")
print(summary_df.to_string(index=False))


Final summary written: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_floor\ipa_summary_approach_1_avoid_step_floor.csv
   P%  IPA_Avg_64  STD_64  IPA_Avg_1024  STD_1024  IPA_Avg_60000  STD_60000
  0.0    0.050473     NaN      0.066880       NaN       0.073994        NaN
 10.0    0.046512     NaN      0.062390       NaN       0.068520        NaN
 20.0    0.043107     NaN      0.056521       NaN       0.063717        NaN
 30.0    0.039116     NaN      0.051599       NaN       0.059506        NaN
 40.0    0.034504     NaN      0.046217       NaN       0.051140        NaN
 50.0    0.029249     NaN      0.039093       NaN       0.044633        NaN
 60.0    0.024239     NaN      0.033054       NaN       0.036322        NaN
 70.0    0.019691     NaN      0.026618       NaN       0.029802        NaN
 80.0    0.014488     NaN      0.019744       NaN       0.021562        NaN
 82.0    0.013348     NaN      0.018387       NaN

In [24]:
# === Cell 4 — Plot IPA vs P% ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TAG   = "1_avoid_step_floor"
TITLE = "Approach 1 (avoid-step, floor asymptote) — Fit on averaged raw CE"
BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

plt.rcParams.update({"font.size": 14})
fig, ax = plt.subplots(figsize=(10, 6))

for bs in BATCH_SIZES:
    mean_col = f"IPA_Avg_{bs}"
    sub = summary_df.dropna(subset=[mean_col])
    if sub.empty:
        continue
    ax.plot(sub["P%"].values, sub[mean_col].values,
            label=f"BS={bs}", color=BS_COLOR[bs], marker="o", markersize=6, linewidth=2)

ax.set_xlabel("Pruning Percentage (%)")
ax.set_ylabel("IPA")
ax.set_title(TITLE, fontsize=14)
ax.grid(True, which="both", alpha=0.3)
ax.legend(frameon=False)

out_png = os.path.join(OUT_DIR, f"ipa_plot_approach_{TAG}.png")
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_floor\ipa_plot_approach_1_avoid_step_floor.png
